# 08 — Training Data Generation

**Pipeline position:** After pipeline (07), before fine-tuning (09-10).

**Purpose:** Generate QA pairs from indexed chunks for reranker and LLM fine-tuning.

**Learning objectives:**
- Generate QA pairs from textbook passages using the local LLM
- Create paraphrase variants and keywords
- Build reranker triplets (query, positive, hard negative)
- Build SFT data in ChatML format for Llama fine-tuning
- Spot-check generated data quality

## Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from src import constant
from src.client.mongodb_config import MongoConfig
from train.build_train_data import (
    generate_qa_pairs,
    generate_paraphrases,
    generate_keywords,
    generate_hard_negative,
    _parse_json_list,
)

## Load chunks from MongoDB

In [ ]:
collection = MongoConfig.get_collection(constant.mongo_collection)
all_chunks = list(collection.find({}, {'_id': 0}))
print(f'Total chunks: {len(all_chunks)}')

# Pick a sample chunk with enough text
sample_chunks = [c for c in all_chunks if len(c.get('page_content', '').split()) > 50]
sample = random.choice(sample_chunks) if sample_chunks else all_chunks[0]
print(f'\nSample chunk ({len(sample["page_content"].split())} words):')
print(sample['page_content'][:300] + '...')

## Generate QA pairs from a single chunk

The LLM generates 5 question-answer pairs from the passage.

In [ ]:
qa_pairs = generate_qa_pairs(sample, n_qa=3)
print(f'Generated {len(qa_pairs)} QA pairs:\n')
for i, qa in enumerate(qa_pairs, 1):
    print(f'[{i}] Q: {qa["question"]}')
    print(f'    A: {qa["answer"][:150]}...')
    print(f'    Source: {qa.get("source_file", "?")} | Chapter: {qa.get("chapter", "?")}\n')

## Generate paraphrases

Each question gets multiple paraphrase variants for training data diversity.

In [ ]:
if qa_pairs:
    paras = generate_paraphrases(qa_pairs[0], n_para=3)
    print(f'Original: {qa_pairs[0]["question"]}\n')
    print('Paraphrases:')
    for j, p in enumerate(paras, 1):
        print(f'  {j}. {p}')

## Build reranker triplet example

Each triplet has: (query, positive_passage, hard_negative_passage).
Hard negatives are topically related but don't answer the question.

In [ ]:
if qa_pairs:
    neg = generate_hard_negative(qa_pairs[0], all_chunks)
    print('=== Reranker Triplet ===')
    print(f'Query:    {qa_pairs[0]["question"]}')
    print(f'Positive: {qa_pairs[0]["passage"][:120]}...')
    print(f'Negative: {neg[:120] if neg else "(none)"}...')

## Build SFT data example (ChatML format)

SFT records use the messages format expected by Llama fine-tuning.

In [ ]:
if qa_pairs:
    sft_example = {
        'messages': [
            {'role': 'system', 'content': 'You are an expert immunology assistant.'},
            {'role': 'user', 'content': f'Context:\n{qa_pairs[0]["passage"][:200]}...\n\nQuestion: {qa_pairs[0]["question"]}'},
            {'role': 'assistant', 'content': qa_pairs[0]['answer']},
        ]
    }
    print(json.dumps(sft_example, indent=2, ensure_ascii=False)[:600])

## Spot-check data quality

Always review generated training data before saving.

In [ ]:
from pathlib import Path

sft_path = Path(constant.train_dir) / 'sft_train.jsonl'
if sft_path.exists():
    with open(sft_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    print(f'SFT training set: {len(lines)} records')
    samples = random.sample(lines, min(3, len(lines)))
    for i, line in enumerate(samples, 1):
        rec = json.loads(line)
        msgs = rec['messages']
        print(f'\n[{i}] Q: {msgs[1]["content"][:100]}...')
        print(f'    A: {msgs[2]["content"][:100]}...')
else:
    print('No SFT data yet. Run: python -m train.build_train_data')

## Summary

**Outputs produced:**
- `data/train/reranker_train.jsonl` — reranker training triplets
- `data/train/reranker_test.jsonl` — reranker test set
- `data/train/sft_train.jsonl` — LLM SFT data in ChatML format
- `data/train/eval_qa.jsonl` — evaluation QA pairs

**Next:** `09_train_reranker.ipynb` — Fine-tune the BGE reranker.